# Dependencies

In [46]:
from datasets import load_dataset
from dotenv import load_dotenv
import os
from pathlib import Path
import torch as T
import torch.nn.functional as F
import string

# Data
Importing the data

In [47]:
load_dotenv("./.env.secrets")

hf_token = os.getenv("HF_TOKEN")

FORMAT = "torch"

ds = load_dataset(
    "Salesforce/wikitext",
    "wikitext-103-v1",
    token=hf_token if hf_token else False,
).with_format(FORMAT)

x_test, x_train, x_val = (ds[key] for key in ds.keys())

display(x_train["text"][:10])

['',
 ' = Valkyria Chronicles III = \n',
 '',
 ' Senjō no Valkyria 3 : <unk> Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " <unk> Raven " . \n',
 " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving

# One hot encoding words

In [48]:
class OneHotEncoder:
    def __init__(self, vocabulary: list[str] | T.Tensor):

        self.dim = len(vocabulary)

        self.word_indexes = {word: i for i, word in enumerate(vocabulary)}

    def encode(self, word: str):
        vec = T.zeros(self.dim)
        idx = self.word_indexes.get(word)
        if idx:
            vec[idx] = 1
        return vec

In [ ]:
def create_vocabulary(data: T.utils.data.Dataset):
    # We clean the words in the dataset a bit, this is to ensure that all words present
    # in the vocabulary are in a predictable format
    to_remove = (
        string.punctuation + "“”‘’"
    )  # Define a set of characters that are to be removed
    table = str.maketrans("", "", to_remove)  # Create a translation table
    big_blob = " ".join(
        data["text"]
    )  # Blob the list of strings into one big string, this consumes a lot of RAM
    cleaned_blob = big_blob.translate(
        table
    )  # Translate the the blob using the table
    return sorted(list(set(cleaned_blob.split())))

In [ ]:
FN = "one_hot_encoder.pt"
file_path = Path(FN)
if file_path.exists():
    encoder = T.load(file_path, weights_only=False)
else:
    vocabulary = create_vocabulary(x_train)
    encoder = OneHotEncoder(vocabulary)
    T.save(encoder, file_path)

tensor([0., 0., 0.,  ..., 0., 0., 0.])


# Embedding model

In [ ]:
class Embedding:
    def __init__(self, dim_in, dim_out):
        self.weights = T.ones((dim_in, dim_out), requires_grad=True)

    def embed(self, vec: T.Tensor):
        T.softmax(vec.T @ self.weights)